<a href="https://colab.research.google.com/github/HVDEER/google_colab/blob/main/Copy_of_cvst_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rsna_intracranial_hemorrhage_detection_path = kagglehub.competition_download('rsna-intracranial-hemorrhage-detection')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_sample_submission.csv
/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection/stage_2_train.csv


In [ ]:
import kagglehub

print("🔄 Bypassing UI... Requesting competition dataset via backend API...")
try:
    # This forces Kaggle to download/mount the competition files via code
    path = kagglehub.competition_download("rsna-intracranial-hemorrhage-detection")
    print(f"🎉 Success! Dataset downloaded and cached at:\n{path}")

    # Let's see what is inside that directory
    import os
    print("\nDirectory contents:")
    print(os.listdir(path))
except Exception as e:
    print(f"❌ Error downloading: {e}")

In [ ]:
import os
print(os.listdir('/kaggle/input'))

import os
path = '/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection'
print(os.listdir(path))

In [ ]:
import pandas as pd
import os

# Path setup
base_path = '/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection'
csv_path = os.path.join(base_path, 'stage_2_train.csv')

print("🚀 Extracting labels and sorting healthy brain slices...")

# 1. Load the labels CSV
df = pd.read_csv(csv_path)

# 2. Split ID into Image_ID and Subtype
df[['Image_ID', 'Subtype']] = df['ID'].str.rsplit('_', n=1, expand=True)

# 🔥 THE FIX: Drop duplicate entries based on Image_ID and Subtype combinations
df = df.drop_duplicates(subset=['Image_ID', 'Subtype'])

# 3. Pivot the table safely now
df_pivoted = df.pivot(index='Image_ID', columns='Subtype', values='Label').reset_index()

# 4. Filter for completely HEALTHY brain slices (where 'any' signs of bleeding == 0)
healthy_scans = df_pivoted[df_pivoted['any'] == 0]

print(f"\n✅ Total Unique Slices in Dataset: {len(df_pivoted):,}")
print(f"✅ Total Healthy Slices Isolated: {len(healthy_scans):,}")

# 5. Randomly sample 1,000 completely healthy scans for our baseline training
train_subset = healthy_scans.sample(n=1000, random_state=42)

# 6. Grab a distinct set of 100 slices for testing later
test_healthy_subset = healthy_scans.drop(train_subset.index).sample(n=100, random_state=42)

print("\n--- Data Subsets Successfully Created! ---")
print(f"Training Baseline (Healthy): {train_subset.shape[0]} slices locked in.")
print(f"Testing Control (Healthy): {test_healthy_subset.shape[0]} slices locked in.")

In [ ]:
import pydicom  # Library for reading medical DICOM files
import matplotlib.pyplot as plt
import numpy as np
import os

def load_and_window_ct(image_id, base_path, window_center=40, window_width=80):
    """
    Loads a raw DICOM CT slice and applies specific windowing (Brain Window)
    to maximize the contrast of soft cerebrovascular tissues.
    """
    # 1. Construct the absolute file path
    file_path = os.path.join(base_path, 'stage_2_train', f"{image_id}.dcm")

    # 2. Read the raw DICOM file
    dicom = pydicom.dcmread(file_path)
    image = dicom.pixel_array.astype(float)

    # 3. Apply Rescale Slope and Intercept (converts raw scanner values to Hounsfield Units)
    slope = float(dicom.RescaleSlope) if 'RescaleSlope' in dicom else 1.0
    intercept = float(dicom.RescaleIntercept) if 'RescaleIntercept' in dicom else 0.0
    hu_image = image * slope + intercept

    # 4. Apply Windowing (Brain Window settings)
    # This clips anything outside our desired density range to preserve brain detail
    img_min = window_center - window_width // 2
    img_max = window_center + window_width // 2

    windowed_image = np.clip(hu_image, img_min, img_max)

    # 5. Min-Max Normalization (Scales pixels strictly between 0.0 and 1.0 for the AI)
    windowed_image = (windowed_image - img_min) / (img_max - img_min)

    return hu_image, windowed_image

# --- Let's visualize a sample from our healthy training subset ---
# Pick the very first image ID from our freshly sorted train_subset
sample_id = train_subset.iloc[0]['Image_ID']
data_directory = '/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection'

try:
    raw_hu, clean_brain = load_and_window_ct(sample_id, data_directory)

    # Plotting both side-by-side to show the power of windowing
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(raw_hu, cmap='gray')
    axes[0].set_title(f"Raw Scanner Data (Unwindowed)\nID: {sample_id}")
    axes[0].axis('off')

    axes[1].imshow(clean_brain, cmap='bone')
    axes[1].set_title("Clinical 'Brain Window' (0-80 HU)\nReady for Anomaly Detection")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
    print("🎉 Pipeline working flawlessly! Scan rendered successfully.")

except Exception as e:
    print(f"❌ Error rendering scan: {e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def inject_scanner_artifact(image, noise_sigma=0.03):
    """
    Simulates high-frequency machine static/thermal noise
    using a global Gaussian distribution.
    """
    noise = np.random.normal(0, noise_sigma, image.shape)
    noisy_image = image + noise
    return np.clip(noisy_image, 0.0, 1.0)

def inject_micro_ischemia(image, x_coord=None, y_coord=None, size=4, attenuation=0.15):
    """
    Simulates an early-stage CVST micro-infarct: a localized, sub-voxel
    structural drop in density near the cerebral venous pathways.
    """
    ischemic_image = image.copy()
    h, w = image.shape

    # If no coordinates are given, pick a spot near the upper sagittal sinus (outer boundary)
    if x_coord is None or y_coord is None:
        x_coord = int(w * 0.5)
        y_coord = int(h * 0.18) # Near the superior sagittal sinus boundary

    # Create a small, slightly blurred binary mask for a natural biological edge
    mask = np.zeros((h, w), dtype=np.float32)
    mask[y_coord:y_coord+size, x_coord:x_coord+size] = 1.0
    mask = cv2.GaussianBlur(mask, (3, 3), 0)

    # Apply attenuation: reduce pixel values by the attenuation factor only within the mask
    ischemic_image = ischemic_image * (1.0 - attenuation * mask)

    return np.clip(ischemic_image, 0.0, 1.0), (x_coord, y_coord, size)

# --- Execute Simulation and Render Results ---
# Reuse our clean_brain scan loaded from the previous step
try:
    # 1. Generate the two artificial classes
    artifact_scan = inject_scanner_artifact(clean_brain, noise_sigma=0.04)
    ischemic_scan, target_box = inject_micro_ischemia(clean_brain, size=5, attenuation=0.15)

    # 2. Plotting the simulation triage
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(clean_brain, cmap='bone')
    axes[0].set_title("1. Baseline Control\n(Perfectly Healthy Brain)", fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(artifact_scan, cmap='bone')
    axes[1].set_title("2. Class A: Scanner Artifact\n(Global Electronic Static)", fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(ischemic_scan, cmap='bone')
    axes[2].set_title("3. Class B: True Micro-Ischemia\n(Subtle Localized Voxel Drop)", fontsize=12)
    # Add a subtle circle indicator to show where the speck was injected for verification
    x, y, s = target_box
    rect = plt.Rectangle((x-5, y-5), s+10, s+10, linewidth=1, edgecolor='r', facecolor='none', linestyle='--')
    axes[2].add_patch(rect)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    print(f"🎉 Simulation matrix built! Injected micro-infarct target coordinates: X={x}, Y={y}")

except Exception as e:
    print(f"❌ Simulation rendering failed: {e}. Make sure 'clean_brain' exists from the previous step.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pydicom
import numpy as np
import os
import cv2

class BrainVolumeDataset(Dataset):
    def __init__(self, dataframe, base_path, mode='train', transform=None):
        """
        Custom PyTorch Dataset for loading and transforming medical DICOM slices.
        """
        self.df = dataframe.reset_index(drop=True)
        self.base_path = base_path
        self.mode = mode
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.iloc[idx]['Image_ID']
        file_path = os.path.join(self.base_path, 'stage_2_train', f"{image_id}.dcm")

        try:
            # 1. Read DICOM file
            dicom = pydicom.dcmread(file_path)
            image = dicom.pixel_array.astype(float)

            # 2. Rescale to Hounsfield Units
            slope = float(dicom.RescaleSlope) if 'RescaleSlope' in dicom else 1.0
            intercept = float(dicom.RescaleIntercept) if 'RescaleIntercept' in dicom else 0.0
            hu_image = image * slope + intercept

            # 3. Apply Brain Window (Center=40, Width=80)
            img_min, img_max = 0, 80
            windowed = np.clip(hu_image, img_min, img_max)
            normalized = (windowed - img_min) / (img_max - img_min)

            # 4. Resize to a standard 256x256 tensor for deep learning efficiency
            normalized = cv2.resize(normalized, (256, 256), interpolation=cv2.INTER_AREA)

        except Exception as e:
            # Fallback placeholder tensor if a file corrupts
            normalized = np.zeros((256, 256), dtype=np.float32)

        # 5. Handle conditional generation based on training vs testing phase
        if self.mode == 'train':
            # Training only receives pristine, healthy baseline images
            img_tensor = torch.tensor(normalized, dtype=torch.float32).unsqueeze(0) # (1, 256, 256)
            return img_tensor, img_tensor # (Input, Target) are identical for Autoencoders!

        elif self.mode == 'test_artifact':
            # Inject global machine static
            noisy = np.random.normal(0, 0.04, normalized.shape) + normalized
            noisy = np.clip(noisy, 0.0, 1.0)
            return torch.tensor(noisy, dtype=torch.float32).unsqueeze(0), torch.tensor(normalized, dtype=torch.float32).unsqueeze(0)

        elif self.mode == 'test_ischemia':
            # Inject our localized structural micro-infarct near the sagittal sinus boundary
            h, w = normalized.shape
            x_coord, y_coord = int(w * 0.5), int(h * 0.18)
            mask = np.zeros((h, w), dtype=np.float32)
            mask[y_coord:y_coord+5, x_coord:x_coord+5] = 1.0
            mask = cv2.GaussianBlur(mask, (3, 3), 0)

            ischemic = normalized * (1.0 - 0.15 * mask)
            ischemic = np.clip(ischemic, 0.0, 1.0)
            return torch.tensor(ischemic, dtype=torch.float32).unsqueeze(0), torch.tensor(normalized, dtype=torch.float32).unsqueeze(0)

# --- Instantiate PyTorch DataLoaders ---
data_directory = '/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection'

# Build training dataset (1,000 baseline healthy images)
train_dataset = BrainVolumeDataset(train_subset, data_directory, mode='train')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

# Build testing datasets (100 images each for checking performance)
test_artifact_dataset = BrainVolumeDataset(test_healthy_subset, data_directory, mode='test_artifact')
test_ischemia_dataset = BrainVolumeDataset(test_healthy_subset, data_directory, mode='test_ischemia')

test_artifact_loader = DataLoader(test_artifact_dataset, batch_size=16, shuffle=False)
test_ischemia_loader = DataLoader(test_ischemia_dataset, batch_size=16, shuffle=False)

# Quick verification check
sample_batch, _ = next(iter(train_loader))
print(f"✅ DataLoaders initialized successfully!")
print(f"📊 Training Batch Shape: {sample_batch.shape} -> (Batch Size, Channels, Height, Width)")

In [ ]:
import torch
import torch.nn as nn

class MedicalAnomalyAutoencoder(nn.Module):
    def __init__(self):
        super(MedicalAnomalyAutoencoder, self).__init__()

        # --- ENCODER: Compress the anatomy ---
        # Input: (1, 256, 256)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),  # -> (16, 128, 128)
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # -> (32, 64, 64)
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # -> (64, 32, 32)
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        # --- DECODER: Reconstruct the anatomy ---
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # -> (32, 64, 64)
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # -> (16, 128, 128)
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1),  # -> (1, 256, 256)
            nn.Sigmoid() # Constrains final pixels strictly between 0.0 and 1.0
        )

    def forward(self, x):
        latent_space = self.encoder(x)
        reconstruction = self.decoder(latent_space)
        return reconstruction

# --- Instantiate and Mount to active Kaggle GPU ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MedicalAnomalyAutoencoder().to(device)

print(f"✅ Neural Network Compiled Successfully!")
print(f"🚀 Active Compute Engine: {device.type.upper()}")

# Quick dry run to verify tensor flow
dummy_input = torch.randn(1, 1, 256, 256).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"📊 Dimensional Flow Check: Input {dummy_input.shape} -> Output {dummy_output.shape}")

In [ ]:
import torch.optim as optim
import time

# 1. Define Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Configure Training Parameters
num_epochs = 5  # Quick training cycle to establish our baseline pipeline
print("🏋️ Starting Neural Network Optimization Loop...")
print("--------------------------------------------------")

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move tensors to active Kaggle GPU
        inputs = inputs.to(device)
        targets = targets.to(device)

        # Zero out gradients from the previous iteration
        optimizer.zero_grad()

        # Forward Pass: Compute reconstructions
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # Backward Pass: Calculate gradients and update weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    elapsed_time = time.time() - start_time
    print(f"Epoch [{epoch+1}/{num_epochs}] -> Loss: {epoch_loss:.6f} | Execution Time: {elapsed_time:.2f}s")

print("--------------------------------------------------")
print("🎉 Baseline Optimization Complete! The network has locked in its structural memory.")

In [ ]:
import matplotlib.pyplot as plt
import torch

# Put the model into evaluation mode
model.eval()

# Grab a single sample batch from both anomaly test loaders
artifact_inputs, _ = next(iter(test_artifact_loader))
ischemia_inputs, _ = next(iter(test_ischemia_loader))

# Move samples to GPU
artifact_inputs = artifact_inputs.to(device)
ischemia_inputs = ischemia_inputs.to(device)

with torch.no_grad():
    # Pass the anomalous scans through our trained network
    artifact_reconstructions = model(artifact_inputs)
    ischemia_reconstructions = model(ischemia_inputs)

# Move everything back to CPU for numpy plotting
art_in = artifact_inputs[0].cpu().squeeze().numpy()
art_out = artifact_reconstructions[0].cpu().squeeze().numpy()
isc_in = ischemia_inputs[0].cpu().squeeze().numpy()
isc_out = ischemia_reconstructions[0].cpu().squeeze().numpy()

# 🧠 CALCULATE RESIDUAL ERROR MAPS (Absolute Difference)
art_residual = np.abs(art_in - art_out)
isc_residual = np.abs(isc_in - isc_out)

# --- Plotting the Final Diagnostic Panel ---
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Global Scanner Artifact Validation
axes[0, 0].imshow(art_in, cmap='bone')
axes[0, 0].set_title("Class A: Input Image\n(Global Electronic Static)")
axes[0, 0].axis('off')

axes[0, 1].imshow(art_out, cmap='bone')
axes[0, 1].set_title("Class A: Autoencoder Output\n(Anatomy Cleaned & Reconstructed)")
axes[0, 1].axis('off')

axes[0, 2].imshow(art_residual, cmap='hot')
axes[0, 2].set_title("Class A: Residual Error Map\n(No Concentrated Structural Spikes)")
axes[0, 2].axis('off')

# Row 2: True Clinical Micro-Ischemia Validation
axes[1, 0].imshow(isc_in, cmap='bone')
axes[1, 0].set_title("Class B: Input Image\n(Hidden Structural Micro-Ischemia)")
axes[1, 0].axis('off')

axes[1, 1].imshow(isc_out, cmap='bone')
axes[1, 1].set_title("Class B: Autoencoder Output\n(Anatomy Reconstructed / Speck 'Healed')")
axes[1, 1].axis('off')

axes[1, 2].imshow(isc_residual, cmap='hot')
axes[1, 2].set_title("Class B: Residual Error Map\n(🔥 ANOMALY DETECTED!)")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()
print("🎉 Framework validation complete! Analyze the Residual Error Maps above.")